In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.multi_modal_llms.openai import OpenAIMultiModal
from llama_index.core.schema import TextNode, ImageNode
import json
import yaml
from llama_index.core.prompts import RichPromptTemplate
from llama_index.core.llms import ChatMessage, MessageRole
import base64

In [2]:
storage_context = StorageContext.from_defaults(persist_dir="./storage")
index = load_index_from_storage(storage_context)

In [37]:
openai_api_key = os.environ["OPENAI_API_KEY"]
client = OpenAI(api_key=openai_api_key)
retriever = index.as_retriever(similarity_top_k=3, image_similarity_top_k=3)
test_query = "Roughly how many parameters did 152 layers have?"
retrieval_results = retriever.retrieve(test_query)
results = type(retrieval_results[0].node)
print(results)

<class 'llama_index.core.schema.ImageNode'>


In [48]:
def encode_image(img_path):
    with open(img_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def retrieval():
    openai_api_key = os.environ["OPENAI_API_KEY"]
    client = OpenAI(api_key=openai_api_key)
    input_question = input("What is your question?")
    retriever = index.as_retriever(similarity_top_k=1, image_similarity_top_k=1)
    retrieval_results = retriever.retrieve(input_question)
    result = retrieval_results[0]
    if type(result.node).__name__ == "TextNode":
        context = result.text
        question = input_question
        qc = f"context:{context},question:{question}"
        with open("textInterpretation.prompt", "r") as f:
            textInterpretationPrompt = f.read()
        resp = client.responses.create(
            model="gpt-5.2",
            input=[
                {"role": "developer", "content": textInterpretationPrompt},
                {"role": "user", "content": qc},
            ],
        )
        answer = resp.output_text
    elif type(result.node).__name__ == "ImageNode":
        path = result.node.image_path
        context = encode_image(path)
        question = input_question
        qc = f"context:(Look at the attached encoded image),question:{question}"
        with open("imageInterpretation.prompt", "r") as f:
            imageInterpretationPrompt = f.read()
        resp = client.chat.completions.create(
            model="gpt-5.2",
            messages=[
                {"role": "developer", "content": imageInterpretationPrompt},
                {"role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": qc
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{context}"
                        }
                    }
                ]
                },
            ],
        )
        answer = resp.choices[0].message.content
        evidence = path
    return answer
print(retrieval())

<class 'str'>
{"answer":"60.2M","units":"M"}


In [5]:
openai_api_key = os.environ["OPENAI_API_KEY"]
client = OpenAI(api_key=openai_api_key)
input_question = input("What is your question?")
retriever = index.as_retriever(similarity_top_k=1, image_similarity_top_k=1)
retrieval_results = retriever.retrieve(input_question)
result = retrieval_results[0]
print(result)

Node ID: 260b0455-ee30-43f5-a6c4-a1cdea9477a1
Text: The graph displays the relationship between the number of
training images in the source task (Instagram) and accuracy percentage
for the CUB2011 target task. It illustrates multiple curves,
indicating an overall trend of increasing accuracy as the number of
training images rises, with all curves approaching or exceeding 80%
accuracy at higher im...
Score:  0.788



In [6]:
print(result.metadata)

{'doc_path': 'extracted_data/document_25/doc_25.md', 'ref_id': 'schwartz2019', 'ref_url': 'https://arxiv.org/pdf/1907.10597', 'img_id': 'img-5.jpeg', 'img_uid': 'schwartz2019img-5.jpeg', 'context': 'ity results. Despite the value of this result in showing that the performance of an LSTM does not plateau after only a few hyperparameter trials, fully exploring the potential of other competitive models for a fair comparison is prohibitively expensive.![img-2.jpeg](img-2.jpeg)\n\n![img-3.jpeg](img-3.jpeg)\n\n![img-4.jpeg](img-4.jpeg)\nFigure 3: Diminishing returns of training on more data: object detection accuracy increases linearly as the number of training examples increases exponentially [25].\n\n![img-5.jpeg](img-5.jpeg)\n\nThe topic of massive number of experiments is not as well studied as the first two discussed above. In fact, the number of experiments performed during model construction is often underreported. Nonetheless, evidence for a logarithmic relation exists here as well, 